## 5 (revised). RQ5 — Classifier false negatives: full origin + per-layer NK signals

Returns TextUnits where the classifier predicts **label=0** (no NK) but
≥ 2 independent linguistic layers disagree.

**Columns returned**

| Column | Content |
| --- | --- |
| `tu_id`, `text`, `position` | TextUnit identity |
| `tu_created_at` | TextUnit creation timestamp |
| `clf_confidence` | Classifier confidence on label=0 |
| `layer_count` | How many linguistic layers fired |
| `repo` | Repository full name |
| `parent_type` | Issue \| PullRequest \| Commit \| Comment |
| `parent_ref` | Issue/PR number, or first 8 chars of commit SHA |
| `author` | GitHub login of artefact author |
| `text_role` | title \| body \| commit_message \| comment_body |
| `lexical_markers` | List of `{rule_id, surface, lemma, category, polarity}` |
| `morph_signals` | List of `{category, subcategory, rule_id, surface}` |
| `word_formation_signals` | List of `{category, subcategory, rule_id, surface}` |
| `rhetorical_signals` | List of `{figure_id, family, subtype, surface}` |
| `n_lex`, `n_morph`, `n_wf`, `n_rhet` | Per-layer signal counts |

> **Cross-product note.** Multiple `OPTIONAL MATCH` clauses for different signal
> layers would multiply row counts before `collect()`. This query uses one
> `CALL {}` subquery per layer to keep the collections independent.

> **Comment parents.** `repo` is null when the TextUnit's parent is a `Comment`
> (Comments are not directly CONTAINED by a Repository). Filter with
> `df[df.repo.notna()]` if you want artefact-level units only.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))

from neo4j import GraphDatabase
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from settings import settings

driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password)
)

def q(cypher, **params):
    """Run a Cypher query and return a DataFrame."""
    with driver.session() as s:
        result = s.run(cypher, **params)
        return pd.DataFrame([r.data() for r in result])

print('Connected to Neo4j.')

Connected to Neo4j.


## 1. Node counts per label

Sanity check: are all layers populated?

In [2]:
df_nodes = q("""
    CALL apoc.meta.stats()
    YIELD labels
    UNWIND keys(labels) AS label
    RETURN label, labels[label] AS count
    ORDER BY count DESC
""")
df_nodes.style.background_gradient(subset=['count'], cmap='Blues')

,label,count
0,Signal,1630
1,TextUnit,781
2,ClassifierVerdict,781
3,Commit,369
4,Repository,100
5,Actor,94
6,PullRequest,59
7,Issue,30
8,LexicalMarker,5
9,RhetoricalFigure,1


## 2. Signal counts per layer × category (heatmap)

In [ ]:
df_signals = q("""
    MATCH (s:Signal)
    RETURN s.layer AS layer, s.category AS category, count(*) AS n
    ORDER BY layer, n DESC
""")

if df_signals.empty:
    print('No signals yet — run the pipeline first.')
else:
    pivot = df_signals.pivot_table(index='layer', columns='category', values='n', fill_value=0)
    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=.5)
    ax.set_title('Signal counts: layer × category')
    plt.tight_layout()
    plt.show()
    print(f'Total signals: {df_signals.n.sum():,}')

## 3. Top lexical markers (RQ1)

Which NK lexemes recur most in the corpus?

In [ ]:
df_markers = q("""
    MATCH (s:Signal {layer:'lexical'})-[:MATCHES_MARKER]->(m:LexicalMarker)
    RETURN m.lemma AS lemma, m.category AS category,
           m.polarity AS polarity, count(s) AS occurrences
    ORDER BY occurrences DESC LIMIT 30
""")
df_markers

In [ ]:
if not df_markers.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = df_markers['polarity'].map({
        'pejorative': '#e74c3c', 'neutral': '#3498db', 'valorising': '#2ecc71'
    }).fillna('#95a5a6')
    ax.barh(df_markers['lemma'], df_markers['occurrences'], color=colors)
    ax.invert_yaxis()
    ax.set_xlabel('Occurrences')
    ax.set_title('Top lexical NK markers (red=pejorative, blue=neutral, green=valorising)')
    plt.tight_layout()
    plt.show()

In [3]:
CYPHER_RQ5 = """
// RQ5 — Classifier false negatives
// TextUnits: classifier label=0 AND >= $min_layers linguistic layers firing.

MATCH (u:TextUnit)-[:CLASSIFIED_AS]->(v:ClassifierVerdict {label: 0})
MATCH (u)-[:HAS_SIGNAL]->(s_filter:Signal)
WHERE s_filter.layer IN ['lexical','morpho_syntactic','word_formation','rhetorical']
WITH u, v, count(DISTINCT s_filter.layer) AS layer_count
WHERE layer_count >= $min_layers

// ── Origin ────────────────────────────────────────────────────────────────
MATCH (parent)-[ht:HAS_TEXT]->(u)
OPTIONAL MATCH (repo:Repository)-[:CONTAINS]->(parent)
OPTIONAL MATCH (actor:Actor)-[:AUTHORED]->(parent)

// ── NK signals per layer (CALL {} avoids cross-product) ───────────────────
CALL {
    WITH u
    OPTIONAL MATCH (u)-[:HAS_SIGNAL]->(sl:Signal {layer: 'lexical'})
    OPTIONAL MATCH (sl)-[:MATCHES_MARKER]->(m:LexicalMarker)
    RETURN collect(DISTINCT {
        rule_id:  sl.rule_id,
        surface:  sl.surface_form,
        lemma:    m.lemma,
        category: m.category,
        polarity: m.polarity
    }) AS lexical_markers
}
CALL {
    WITH u
    OPTIONAL MATCH (u)-[:HAS_SIGNAL]->(sm:Signal {layer: 'morpho_syntactic'})
    RETURN collect(DISTINCT {
        category:    sm.category,
        subcategory: sm.subcategory,
        rule_id:     sm.rule_id,
        surface:     sm.surface_form
    }) AS morph_signals
}
CALL {
    WITH u
    OPTIONAL MATCH (u)-[:HAS_SIGNAL]->(sw:Signal {layer: 'word_formation'})
    RETURN collect(DISTINCT {
        category:    sw.category,
        subcategory: sw.subcategory,
        rule_id:     sw.rule_id,
        surface:     sw.surface_form
    }) AS word_formation_signals
}
CALL {
    WITH u
    OPTIONAL MATCH (u)-[:HAS_SIGNAL]->(sr:Signal {layer: 'rhetorical'})
    OPTIONAL MATCH (sr)-[:INSTANTIATES]->(f:RhetoricalFigure)
    RETURN collect(DISTINCT {
        figure_id: f.figure_id,
        family:    f.family,
        subtype:   f.subtype,
        surface:   sr.surface_form
    }) AS rhetorical_signals
}

RETURN
    // Identity
    u.id              AS tu_id,
    u.text            AS text,
    u.position        AS position,
    u.created_at      AS tu_created_at,

    // Classifier
    v.confidence      AS clf_confidence,
    layer_count,

    // Origin
    coalesce(repo.full_name, '(comment)')  AS repo,
    labels(parent)[0]                      AS parent_type,
    CASE labels(parent)[0]
        WHEN 'Issue'       THEN toString(parent.number)
        WHEN 'PullRequest' THEN toString(parent.number)
        WHEN 'Commit'      THEN left(parent.sha, 8)
        ELSE null
    END                                    AS parent_ref,
    actor.login                            AS author,
    ht.role                                AS text_role,

    // NK signals — each layer in its own column
    lexical_markers,
    morph_signals,
    word_formation_signals,
    rhetorical_signals,

    // Flat counts for quick filtering
    size(lexical_markers)        AS n_lex,
    size(morph_signals)          AS n_morph,
    size(word_formation_signals) AS n_wf,
    size(rhetorical_signals)     AS n_rhet

ORDER BY layer_count DESC, clf_confidence DESC
LIMIT $limit
"""

df_fn = q(CYPHER_RQ5, min_layers=2, limit=50)
print(f"False negatives found: {len(df_fn)}")
df_fn[[
    'repo', 'parent_type', 'parent_ref', 'author', 'text_role',
    'tu_created_at', 'position', 'clf_confidence', 'layer_count',
    'n_lex', 'n_morph', 'n_wf', 'n_rhet', 'text'
]].head(20)

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (u) { ... }', position=<SummaryInputPosition line=17, column=1, offset=702>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 702, 'line': 17, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n// RQ5 — Classifier false negatives\n// TextUnits: classifier label=0 AND >= $min_layers linguistic layers firing.\n\nMATCH (u:TextUnit)-[:CLASSIFIED_AS]->(v:ClassifierVerdict {label: 0})\nMATCH (u)-[:HAS_SIGNAL]->(s_filter:Signal)\nWHERE s_filter.layer IN ['lexical','morpho_syntactic','word_formation','rhetorical']\nWITH u, v, count(DI

False negatives found: 50


,repo,parent_type,parent_ref,author,text_role,tu_created_at,position,clf_confidence,layer_count,n_lex,n_morph,n_wf,n_rhet,text
0,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."
1,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."
2,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."
3,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."
4,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."
5,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."
6,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."
7,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."
8,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."
9,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,7,0.984375,3,1,2,1,1,"This is green now, just need to get some clari..."


In [4]:
# ── Flatten nested signal lists into readable columns ─────────────────────
# Each *_signals column is a list of dicts.  These helpers extract flat lists
# of the most useful fields for quick scanning.

def pick(lst, key):
    """Extract `key` from each dict in a list, dropping None values."""
    return [item[key] for item in (lst or []) if item.get(key) is not None]

if not df_fn.empty:
    df_fn['lex_lemmas']     = df_fn['lexical_markers'].apply(lambda x: pick(x, 'lemma'))
    df_fn['lex_polarities'] = df_fn['lexical_markers'].apply(lambda x: pick(x, 'polarity'))
    df_fn['morph_cats']     = df_fn['morph_signals'].apply(
                                  lambda x: list(dict.fromkeys(pick(x, 'category'))))
    df_fn['morph_subcats']  = df_fn['morph_signals'].apply(
                                  lambda x: list(dict.fromkeys(pick(x, 'subcategory'))))
    df_fn['rhet_families']  = df_fn['rhetorical_signals'].apply(lambda x: pick(x, 'family'))
    df_fn['rhet_subtypes']  = df_fn['rhetorical_signals'].apply(lambda x: pick(x, 'subtype'))
    df_fn['wf_rules']       = df_fn['word_formation_signals'].apply(lambda x: pick(x, 'rule_id'))

    display(df_fn[[
        'repo', 'parent_type', 'parent_ref', 'author', 'text_role', 'tu_created_at',
        'clf_confidence', 'layer_count',
        'lex_lemmas', 'lex_polarities',
        'morph_cats', 'morph_subcats',
        'wf_rules',
        'rhet_families', 'rhet_subtypes',
        'text'
    ]].head(20))

,repo,parent_type,parent_ref,author,text_role,tu_created_at,clf_confidence,layer_count,lex_lemmas,lex_polarities,morph_cats,morph_subcats,wf_rules,rhet_families,rhet_subtypes,text
0,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."
1,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."
2,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."
3,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."
4,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."
5,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."
6,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."
7,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."
8,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."
9,voxpupuli/puppet-puppetserver,PullRequest,46,bastelfreak,comment_body,2017-09-29T23:43:36Z,0.984375,3,[error],[pejorative],"[negation, hedging]","[adverbial_not, approximator]",[affix.prefix.in],[],[],"This is green now, just need to get some clari..."


## 4. Classifier label distribution vs. signal layer-diversity

Scatter: each point is a TextUnit. x = number of distinct linguistic layers
that fired; y = classifier confidence. Colour = classifier label.

Units in the top-left (high confidence, 0 linguistic layers) are classifier
false positives. Units in the bottom-right (many layers, low confidence) are
classifier false negatives. Both are analytically interesting for RQ5.

In [ ]:
df_rq5_raw = q("""
    MATCH (u:TextUnit)-[:CLASSIFIED_AS]->(v:ClassifierVerdict)
    OPTIONAL MATCH (u)-[:HAS_SIGNAL]->(s:Signal)
    WHERE s.layer IN ['lexical','morpho_syntactic','word_formation','rhetorical']
    WITH u, v, count(DISTINCT s.layer) AS layer_diversity
    RETURN v.label AS label,
           v.confidence AS confidence,
           layer_diversity,
           u.id AS tu_id
""")

if df_rq5_raw.empty:
    print('No ClassifierVerdict nodes yet. Run ClassifierAnnotator first.')
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    for label, grp in df_rq5_raw.groupby('label'):
        color = '#e74c3c' if label == 1 else '#3498db'
        ax.scatter(grp['layer_diversity'], grp['confidence'],
                   alpha=0.4, s=15, c=color,
                   label=f'label={label} (n={len(grp):,})')
    ax.set_xlabel('Distinct linguistic layers with ≥1 signal')
    ax.set_ylabel('Classifier confidence')
    ax.set_title('RQ5 scatter: classifier confidence vs. linguistic layer diversity')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5 (revised). RQ5 — Classifier false negatives: full origin + per-layer NK signals

Returns TextUnits where the classifier predicts **label=0** (no NK) but
≥ 2 independent linguistic layers disagree.

**Columns returned**

| Column | Content |
| --- | --- |
| `tu_id`, `text`, `position` | TextUnit identity |
| `tu_created_at` | TextUnit creation timestamp |
| `clf_confidence` | Classifier confidence on label=0 |
| `layer_count` | How many linguistic layers fired |
| `repo` | Repository full name |
| `parent_type` | Issue \| PullRequest \| Commit \| Comment |
| `parent_ref` | Issue/PR number, or first 8 chars of commit SHA |
| `author` | GitHub login of artefact author |
| `text_role` | title \| body \| commit_message \| comment_body |
| `lexical_markers` | List of `{rule_id, surface, lemma, category, polarity}` |
| `morph_signals` | List of `{category, subcategory, rule_id, surface}` |
| `word_formation_signals` | List of `{category, subcategory, rule_id, surface}` |
| `rhetorical_signals` | List of `{figure_id, family, subtype, surface}` |
| `n_lex`, `n_morph`, `n_wf`, `n_rhet` | Per-layer signal counts |

> **Cross-product note.** Multiple `OPTIONAL MATCH` clauses for different signal
> layers would multiply row counts before `collect()`. This query uses one
> `CALL {}` subquery per layer to keep the collections independent.

> **Comment parents.** `repo` is null when the TextUnit's parent is a `Comment`
> (Comments are not directly CONTAINED by a Repository). Filter with
> `df[df.repo.notna()]` if you want artefact-level units only.

In [ ]:
CYPHER_RQ5 = """
// RQ5 — Classifier false negatives
// TextUnits: classifier label=0 AND >= $min_layers linguistic layers firing.

MATCH (u:TextUnit)-[:CLASSIFIED_AS]->(v:ClassifierVerdict {label: 0})
MATCH (u)-[:HAS_SIGNAL]->(s_filter:Signal)
WHERE s_filter.layer IN ['lexical','morpho_syntactic','word_formation','rhetorical']
WITH u, v, count(DISTINCT s_filter.layer) AS layer_count
WHERE layer_count >= $min_layers

// ── Origin ────────────────────────────────────────────────────────────────
MATCH (parent)-[ht:HAS_TEXT]->(u)
OPTIONAL MATCH (repo:Repository)-[:CONTAINS]->(parent)
OPTIONAL MATCH (actor:Actor)-[:AUTHORED]->(parent)

// ── NK signals per layer (CALL {} avoids cross-product) ───────────────────
CALL {
    WITH u
    OPTIONAL MATCH (u)-[:HAS_SIGNAL]->(sl:Signal {layer: 'lexical'})
    OPTIONAL MATCH (sl)-[:MATCHES_MARKER]->(m:LexicalMarker)
    RETURN collect(DISTINCT {
        rule_id:  sl.rule_id,
        surface:  sl.surface_form,
        lemma:    m.lemma,
        category: m.category,
        polarity: m.polarity
    }) AS lexical_markers
}
CALL {
    WITH u
    OPTIONAL MATCH (u)-[:HAS_SIGNAL]->(sm:Signal {layer: 'morpho_syntactic'})
    RETURN collect(DISTINCT {
        category:    sm.category,
        subcategory: sm.subcategory,
        rule_id:     sm.rule_id,
        surface:     sm.surface_form
    }) AS morph_signals
}
CALL {
    WITH u
    OPTIONAL MATCH (u)-[:HAS_SIGNAL]->(sw:Signal {layer: 'word_formation'})
    RETURN collect(DISTINCT {
        category:    sw.category,
        subcategory: sw.subcategory,
        rule_id:     sw.rule_id,
        surface:     sw.surface_form
    }) AS word_formation_signals
}
CALL {
    WITH u
    OPTIONAL MATCH (u)-[:HAS_SIGNAL]->(sr:Signal {layer: 'rhetorical'})
    OPTIONAL MATCH (sr)-[:INSTANTIATES]->(f:RhetoricalFigure)
    RETURN collect(DISTINCT {
        figure_id: f.figure_id,
        family:    f.family,
        subtype:   f.subtype,
        surface:   sr.surface_form
    }) AS rhetorical_signals
}

RETURN
    // Identity
    u.id              AS tu_id,
    u.text            AS text,
    u.position        AS position,
    u.created_at      AS tu_created_at,

    // Classifier
    v.confidence      AS clf_confidence,
    layer_count,

    // Origin
    coalesce(repo.full_name, '(comment)')  AS repo,
    labels(parent)[0]                      AS parent_type,
    CASE labels(parent)[0]
        WHEN 'Issue'       THEN toString(parent.number)
        WHEN 'PullRequest' THEN toString(parent.number)
        WHEN 'Commit'      THEN left(parent.sha, 8)
        ELSE null
    END                                    AS parent_ref,
    actor.login                            AS author,
    ht.role                                AS text_role,

    // NK signals — each layer in its own column
    lexical_markers,
    morph_signals,
    word_formation_signals,
    rhetorical_signals,

    // Flat counts for quick filtering
    size(lexical_markers)        AS n_lex,
    size(morph_signals)          AS n_morph,
    size(word_formation_signals) AS n_wf,
    size(rhetorical_signals)     AS n_rhet

ORDER BY layer_count DESC, clf_confidence DESC
LIMIT $limit
"""

df_fn = q(CYPHER_RQ5, min_layers=2, limit=50)
print(f"False negatives found: {len(df_fn)}")
df_fn[[
    'repo', 'parent_type', 'parent_ref', 'author', 'text_role',
    'tu_created_at', 'position', 'clf_confidence', 'layer_count',
    'n_lex', 'n_morph', 'n_wf', 'n_rhet', 'text'
]].head(20)

In [ ]:
# ── Flatten nested signal lists into readable columns ─────────────────────
# Each *_signals column is a list of dicts.  These helpers extract flat lists
# of the most useful fields for quick scanning.

def pick(lst, key):
    """Extract `key` from each dict in a list, dropping None values."""
    return [item[key] for item in (lst or []) if item.get(key) is not None]

if not df_fn.empty:
    df_fn['lex_lemmas']     = df_fn['lexical_markers'].apply(lambda x: pick(x, 'lemma'))
    df_fn['lex_polarities'] = df_fn['lexical_markers'].apply(lambda x: pick(x, 'polarity'))
    df_fn['morph_cats']     = df_fn['morph_signals'].apply(
                                  lambda x: list(dict.fromkeys(pick(x, 'category'))))
    df_fn['morph_subcats']  = df_fn['morph_signals'].apply(
                                  lambda x: list(dict.fromkeys(pick(x, 'subcategory'))))
    df_fn['rhet_families']  = df_fn['rhetorical_signals'].apply(lambda x: pick(x, 'family'))
    df_fn['rhet_subtypes']  = df_fn['rhetorical_signals'].apply(lambda x: pick(x, 'subtype'))
    df_fn['wf_rules']       = df_fn['word_formation_signals'].apply(lambda x: pick(x, 'rule_id'))

    display(df_fn[[
        'repo', 'parent_type', 'parent_ref', 'author', 'text_role', 'tu_created_at',
        'clf_confidence', 'layer_count',
        'lex_lemmas', 'lex_polarities',
        'morph_cats', 'morph_subcats',
        'wf_rules',
        'rhet_families', 'rhet_subtypes',
        'text'
    ]].head(20))

## 6. Signal layer co-occurrence matrix

Which layers tend to fire together? Relevant for RQ2 prototypical co-occurrences.

In [ ]:
layers = ['lexical', 'morpho_syntactic', 'word_formation', 'rhetorical']

cooc = pd.DataFrame(0, index=layers, columns=layers)
for i, l1 in enumerate(layers):
    for l2 in layers[i:]:
        df_pair = q("""
            MATCH (u:TextUnit)-[:HAS_SIGNAL]->(s1:Signal {layer: $l1})
            MATCH (u)-[:HAS_SIGNAL]->(s2:Signal {layer: $l2})
            RETURN count(DISTINCT u) AS n
        """, l1=l1, l2=l2)
        n = int(df_pair['n'].iloc[0]) if not df_pair.empty else 0
        cooc.loc[l1, l2] = n
        cooc.loc[l2, l1] = n

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cooc, annot=True, fmt='d', cmap='Purples', ax=ax)
ax.set_title('TextUnits with co-occurring signals from layer pairs (RQ2)')
plt.tight_layout()
plt.show()

In [ ]:
driver.close()
print('Done.')